In [12]:
import numpy as np
import os
from typing import List, Tuple
from pathlib import Path
from ipynb.fs.full.audio_parser import audio_convert, spectrogram_conversion
from ipynb.fs.full.fingerprint_maker import generate_fingerprints

In [13]:
#need to finish commenting

def create_database(): #highkey redundant if not assigning independent string
    fingerprint_database = dict()
    return fingerprint_database


def add_fingerprints(database: dict, song_id: str, fingerprints: list):
    #adds songs from our library to a dictionary of fingerprints
    for (fm, fn, dt), tm in fingerprints:
        if (fm, fn, dt) not in database:
                database[(fm, fn, dt)] = []
        database[(fm, fn, dt)].append((song_id, tm))

def query_database(database: dict, query_fingerprints: list, freq_tolerance: int = 2, delta_tolerance: int = 2):

    print("Sample Database Keys:", list(database.keys())[:3])
    print("Sample Query Fingerprint:", query_fingerprints[0] if query_fingerprints else "Empty Query")

    match_counts = {}
    """
    Gets the peaks of an audio file
    Shifts it around each file in the database
    Compares the peaks across each audio file at every time window
    returns the matches
    """
    for (fm, fn, dt), tq in query_fingerprints:
        fm, fn, dt, tq = int(fm), int(fn), int(dt), int(tq)
        for df_fm in range(-freq_tolerance, freq_tolerance + 1):
            for df_fn in range(-freq_tolerance, freq_tolerance + 1):
                for d_dt in range(-delta_tolerance, delta_tolerance + 1):
                    fingerprint_key = (fm + df_fm, fn + df_fn, dt + d_dt)

                    if fingerprint_key in database:
                        for song_id, tm in database[fingerprint_key]:
                            time_offset = tm - tq
                            key = (song_id, time_offset)
                            match_counts[key] = match_counts.get(key, 0) + 1

    return match_counts


def get_sorted_matches(match_counts):
   #returns song id and time offset of best match
   #need to add probability feature (if agreed on) using returned highest_count
   #and remove time offset artifact from highest key.
   if len(match_counts) == 0:
       return None
  
   sorted_matches = sorted(match_counts.items(), key=lambda item: item[1], reverse=True)
   return sorted_matches

def get_sorted_songs(sorted_matches, num_recs: int = 5):
    #extracts song ID and sorts them into nonrepeating array of max length num_recs
    sorted_songs = [song_id for ((song_id, time_offset), votes) in sorted_matches]
    sorted_songs = []
    for ((song_id, time_offset), votes) in sorted_matches:
        if len(sorted_songs) == num_recs:
            break
        if song_id in sorted_songs:
            continue
        sorted_songs.append(song_id)
    return sorted_songs


In [ ]:
database = create_database()

for filename in os.listdir("Music"):
    if filename.endswith(".txt"): #can't parse list of songs
        continue
    song_id = os.path.splitext(filename)[0] #removes .wav from the end
    song_path = os.path.join("Music", filename) #same as original code from here
    samples, sr = audio_convert(song_path)
    _, peaks = spectrogram_conversion(samples, sr)
    fps = generate_fingerprints(peaks, fanout=3)
    add_fingerprints(database, song_id=song_id, fingerprints=fps)

# Query with the recorded clip (same code from earlier)
test_samples, test_sample_rate = audio_convert(os.path.join("Recorded Songs", "black milk clear.wav"))
_, test_peaks = spectrogram_conversion(test_samples, test_sample_rate)
test_fingerprints = generate_fingerprints(test_peaks, fanout=3)

match_counts = query_database(database, test_fingerprints, freq_tolerance=2, delta_tolerance=2)
sorted_matches = get_sorted_matches(match_counts)
sorted_songs = get_sorted_songs(sorted_matches)
print("Best match:", sorted_songs[0])
print("Recommended songs:", sorted_songs[1:6]) 

# print("Database fingerprints:", len(my_fingerprints))
print("Test fingerprints:", len(test_fingerprints))
print("Matches:", len(match_counts))

Sample Database Keys: [(70, 5, 1), (70, 250, 1), (70, 395, 1)]
Sample Query Fingerprint: ((42, 12, 11), 1)
Best match: Black Milk
Recommended songs: ['02 - Man I Need', '01 - Die With A Smile', 'Babydoll', 'Lover Girl']
Test fingerprints: 62
Matches: 52
